In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Cluster Heartbeat - Feature Engineering\n",
    "\n",
    "## Feature Extraction and Engineering for GPU Cluster Metrics\n",
    "\n",
    "This notebook demonstrates feature extraction, engineering, and normalization for the Cluster Heartbeat system."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "import os\n",
    "from pathlib import Path\n",
    "\n",
    "sys.path.insert(0, str(Path.cwd().parent))\n",
    "\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from sklearn.decomposition import PCA\n",
    "from sklearn.preprocessing import StandardScaler\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "plt.style.use('seaborn-v0_8-darkgrid')\n",
    "sns.set_palette(\"husl\")\n",
    "%matplotlib inline"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from src.config import load_config\n",
    "from src.data.ingestion import DataIngestion\n",
    "from src.features.extractor import FeatureExtractor\n",
    "from src.features.normalizer import FeatureNormalizer\n",
    "from src.data.preprocessing import DataPreprocessor"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load configuration\n",
    "config = load_config('../config/config.yaml')\n",
    "print(\"Configuration loaded successfully!\")\n",
    "print(f\"Window size: {config['data']['processing']['window_size']} seconds\")\n",
    "print(f\"Stride: {config['data']['processing']['stride']} seconds\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load and preprocess data\n",
    "ingestion = DataIngestion(config)\n",
    "df = ingestion.load_data(source='synthetic')\n",
    "\n",
    "preprocessor = DataPreprocessor(config)\n",
    "df_clean = preprocessor.clean_data(df)\n",
    "\n",
    "print(f\"Data shape: {df_clean.shape}\")\n",
    "df_clean.head()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Initialize feature extractor\n",
    "extractor = FeatureExtractor(config)\n",
    "\n",
    "# Extract windows\n",
    "windows = extractor.extract_windows(df_clean)\n",
    "print(f\"Number of windows: {len(windows)}\")\n",
    "print(f\"Window shape: {windows[0].shape}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Extract features from a single window\n",
    "sample_window = windows[0]\n",
    "features = extractor.extract_features(sample_window)\n",
    "\n",
    "print(f\"Features extracted: {len(features)}\")\n",
    "print(f\"Feature names: {features.get_feature_names()[:5]}...\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Extract all features\n",
    "feature_matrix, window_features = extractor.extract_all_features(df_clean)\n",
    "\n",
    "print(f\"Feature matrix shape: {feature_matrix.shape}\")\n",
    "print(f\"Number of windows: {len(window_features)}\")\n",
    "\n",
    "# Convert to DataFrame for visualization\n",
    "feature_df = pd.DataFrame(feature_matrix)\n",
    "feature_df.head()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize feature distribution\n",
    "fig, axes = plt.subplots(2, 3, figsize=(15, 8))\n",
    "axes = axes.flatten()\n",
    "\n",
    "# Sample first 6 features\n",
    "for i in range(min(6, feature_matrix.shape[1])):\n",
    "    axes[i].hist(feature_matrix[:, i], bins=30, alpha=0.7, edgecolor='black')\n",
    "    axes[i].set_title(f'Feature {i}', fontsize=12)\n",
    "    axes[i].set_xlabel('Value')\n",
    "    axes[i].set_ylabel('Frequency')\n",
    "\n",
    "for j in range(i+1, len(axes)):\n",
    "    axes[j].set_visible(False)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Normalize features\n",
    "normalizer = FeatureNormalizer(config)\n",
    "features_norm = normalizer.fit_transform(feature_matrix)\n",
    "\n",
    "print(f\"Original features shape: {feature_matrix.shape}\")\n",
    "print(f\"Normalized features shape: {features_norm.shape}\")\n",
    "\n",
    "# Check normalization\n",
    "print(f\"\\nNormalized features statistics:\")\n",
    "print(f\"Mean: {np.mean(features_norm):.6f}\")\n",
    "print(f\"Std: {np.std(features_norm):.6f}\")\n",
    "print(f\"Min: {np.min(features_norm):.6f}\")\n",
    "print(f\"Max: {np.max(features_norm):.6f}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Dimensionality reduction with PCA\n",
    "pca = PCA(n_components=0.95)  # Keep 95% variance\n",
    "features_reduced = pca.fit_transform(features_norm)\n",
    "\n",
    "print(f\"Original features: {features_norm.shape[1]}\")\n",
    "print(f\"Reduced features: {features_reduced.shape[1]}\")\n",
    "print(f\"Explained variance: {pca.explained_variance_ratio_.sum():.2%}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize PCA components\n",
    "plt.figure(figsize=(12, 6))\n",
    "\n",
    "# Explained variance\n",
    "plt.subplot(1, 2, 1)\n",
    "plt.bar(range(1, len(pca.explained_variance_ratio_) + 1), \n",
    "        pca.explained_variance_ratio_, alpha=0.7, edgecolor='black')\n",
    "plt.axhline(y=0.05, color='red', linestyle='--', label='5% threshold')\n",
    "plt.xlabel('Principal Component')\n",
    "plt.ylabel('Explained Variance Ratio')\n",
    "plt.title('Explained Variance by Component')\n",
    "plt.legend()\n",
    "\n",
    "# Cumulative variance\n",
    "plt.subplot(1, 2, 2)\n",
    "cumulative_variance = np.cumsum(pca.explained_variance_ratio_)\n",
    "plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, 'b-')\n",
    "plt.axhline(y=0.95, color='red', linestyle='--', label='95% threshold')\n",
    "plt.xlabel('Number of Components')\n",
    "plt.ylabel('Cumulative Explained Variance')\n",
    "plt.title('Cumulative Explained Variance')\n",
    "plt.legend()\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize reduced features\n",
    "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "\n",
    "# 2D projection\n",
    "if features_reduced.shape[1] >= 2:\n",
    "    axes[0].scatter(features_reduced[:, 0], features_reduced[:, 1], \n",
    "                    alpha=0.5, s=20, c=np.random.rand(len(features_reduced)))\n",
    "    axes[0].set_xlabel('PC1')\n",
    "    axes[0].set_ylabel('PC2')\n",
    "    axes[0].set_title('PCA Projection (2D)')\n",
    "\n",
    "# Feature importance\n",
    "if hasattr(pca, 'components_'):\n",
    "    importance = np.abs(pca.components_[0])\n",
    "    top_indices = np.argsort(importance)[-10:]\n",
    "    axes[1].barh(range(len(top_indices)), importance[top_indices], alpha=0.7)\n",
    "    axes[1].set_yticks(range(len(top_indices)))\n",
    "    axes[1].set_yticklabels([f'Feature {i}' for i in top_indices])\n",
    "    axes[1].set_xlabel('Importance')\n",
    "    axes[1].set_title('Top 10 Features by PC1 Importance')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Feature pipeline (complete)\n",
    "from src.features.normalizer import FeaturePipeline\n",
    "\n",
    "pipeline = FeaturePipeline(config)\n",
    "features = pipeline.fit_transform(df_clean)\n",
    "\n",
    "print(f\"Pipeline output shape: {features.shape}\")\n",
    "print(f\"Pipeline fitted: {pipeline.is_fitted}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Feature importance summary\n",
    "from sklearn.ensemble import RandomForestRegressor\n",
    "\n",
    "# Use a sample target (GPU utilization)\n",
    "target = df_clean['gpu_utilization'].values[:len(features)]\n",
    "\n",
    "# Fit random forest\n",
    "rf = RandomForestRegressor(n_estimators=100, random_state=42)\n",
    "rf.fit(features, target)\n",
    "\n",
    "# Get feature importance\n",
    "importance = rf.feature_importances_\n",
    "top_indices = np.argsort(importance)[-20:]\n",
    "\n",
    "plt.figure(figsize=(12, 8))\n",
    "plt.barh(range(len(top_indices)), importance[top_indices], alpha=0.7, edgecolor='black')\n",
    "plt.yticks(range(len(top_indices)), [f'Feature {i}' for i in top_indices])\n",
    "plt.xlabel('Feature Importance')\n",
    "plt.title('Top 20 Features for Predicting GPU Utilization')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Summary\n",
    "print(\"=\"*60)\n",
    "print(\"FEATURE ENGINEERING SUMMARY\")\n",
    "print(\"=\"*60)\n",
    "print(f\"\\n✅ Original features: {feature_matrix.shape[1]}\")\n",
    "print(f\"✅ Reduced features: {features_reduced.shape[1]}\")\n",
    "print(f\"✅ Variance explained: {pca.explained_variance_ratio_.sum():.2%}\")\n",
    "print(f\"✅ Pipeline output: {features.shape}\")\n",
    "print(f\"✅ Top feature importance: {importance[top_indices[-1]]:.3f}\")\n",
    "\n",
    "print(\"\\n📊 Features ready for model training!\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}